# AM5061 · Week 2 · Chilled-water distribution

**Design of Thermal and Fluid Systems** · Applied Mechanics, IIT Madras · Jul–Nov 2026

Run the two setup cells below once, then work down the notebook. Nothing needs to be installed on your own machine.


## Setup

Run these two cells first. The second one writes the course helper module, so this notebook is self-contained.


In [ ]:
#@title Install the property library  { display-mode: "form" }
!pip install -q CoolProp openpyxl
print('CoolProp ready')


In [ ]:
%%writefile am5061.py
"""AM5061 - Design of Thermal and Fluid Systems.
Shared helpers for the course notebooks.

Design of Thermal and Fluid Systems, IIT Madras, Jul-Nov 2026.

This module is deliberately thin. It wraps CoolProp with names and units that
match the lecture notation, adds an Excel writer that produces workbooks you
can actually filter, and sets a consistent plot style. It does NOT hide the
engineering: every case notebook still writes its own equations.

Install (first cell of any Colab notebook):
    !pip install -q CoolProp openpyxl

Units are SI throughout, with ONE exception that is flagged everywhere it
appears: temperatures in function arguments named `..._C` are in Celsius,
because that is how the case briefs state them. Everything internal is kelvin.
"""
from __future__ import annotations

import math

from CoolProp.CoolProp import PropsSI, PhaseSI

__all__ = [
    "K", "C", "State", "state", "sat_liquid", "sat_vapour", "p_sat", "T_sat",
    "h_fg", "critical", "fluids", "solve", "sweep", "to_excel",
    "style_plots", "NAVY", "ORANGE", "BLUE", "MUTED",
]

# ---------------------------------------------------------------- constants
NAVY, ORANGE, BLUE, MUTED = "#1F3864", "#ED7D31", "#4472C4", "#59626E"
T0 = 273.15


def K(t_celsius: float) -> float:
    """Celsius -> kelvin. Use this at the boundary, never inside a formula."""
    return t_celsius + T0


def C(t_kelvin: float) -> float:
    """Kelvin -> Celsius, for reporting only."""
    return t_kelvin - T0


# ------------------------------------------------------------------- states
class State:
    """A thermodynamic state. Immutable, and it knows its own fluid.

    Construct it with any two independent properties:
        State("R134a", P=1e6, T=K(70))
        State("Water", P=101325, Q=0)      # saturated liquid
        State("R134a", P=p_cond, H=h2)

    Then read properties as attributes: .T .p .h .s .d .cp .x
    Attribute names match the lecture notation, not CoolProp's letter codes,
    so a student reading the notebook does not need the CoolProp manual open.
    """

    _MAP = {"T": "T", "P": "P", "H": "H", "S": "S", "D": "D", "Q": "Q"}

    def __init__(self, fluid: str, **kw):
        if len(kw) != 2:
            raise ValueError(
                f"a state needs exactly two properties, got {list(kw)}. "
                "Two and only two - that is the phase rule, not a quirk."
            )
        (n1, v1), (n2, v2) = kw.items()
        for n in (n1, n2):
            if n not in self._MAP:
                raise ValueError(f"unknown property {n!r}; use T, P, H, S, D or Q")
        self.fluid, self._args = fluid, (n1, v1, n2, v2)

    def _get(self, what: str) -> float:
        n1, v1, n2, v2 = self._args
        return PropsSI(what, n1, v1, n2, v2, self.fluid)

    # Named so they read like the equations on the slides.
    T  = property(lambda s: s._get("T"),  doc="temperature, K")
    p  = property(lambda s: s._get("P"),  doc="pressure, Pa")
    h  = property(lambda s: s._get("H"),  doc="specific enthalpy, J/kg")
    s  = property(lambda s: s._get("S"),  doc="specific entropy, J/kg.K")
    d  = property(lambda s: s._get("D"),  doc="density, kg/m3")
    cp = property(lambda s: s._get("C"),  doc="cp, J/kg.K")
    mu = property(lambda s: s._get("V"),  doc="dynamic viscosity, Pa.s")
    k  = property(lambda s: s._get("L"),  doc="thermal conductivity, W/m.K")
    x  = property(lambda s: s._get("Q"),  doc="vapour quality, - (=-1 if single phase)")

    @property
    def T_C(self) -> float:
        return C(self.T)

    @property
    def phase(self) -> str:
        n1, v1, n2, v2 = self._args
        return PhaseSI(n1, v1, n2, v2, self.fluid)

    def __repr__(self):
        try:
            return (f"State({self.fluid}: {self.T_C:.2f} C, {self.p/1e5:.3f} bar, "
                    f"h={self.h/1e3:.2f} kJ/kg, {self.phase})")
        except Exception:
            return f"State({self.fluid}, {self._args})"


def state(fluid: str, **kw) -> State:
    """Shorthand for State(...)."""
    return State(fluid, **kw)


def sat_liquid(fluid: str, *, T=None, p=None) -> State:
    """Saturated liquid at T or p. Give one, not both."""
    if (T is None) == (p is None):
        raise ValueError("give exactly one of T or p")
    return State(fluid, T=T, Q=0) if T is not None else State(fluid, P=p, Q=0)


def sat_vapour(fluid: str, *, T=None, p=None) -> State:
    """Saturated vapour at T or p."""
    if (T is None) == (p is None):
        raise ValueError("give exactly one of T or p")
    return State(fluid, T=T, Q=1) if T is not None else State(fluid, P=p, Q=1)


def p_sat(fluid: str, T: float) -> float:
    """Saturation pressure, Pa. For a BLEND this is the bubble-point pressure."""
    return PropsSI("P", "T", T, "Q", 0, fluid)


def T_sat(fluid: str, p: float) -> float:
    """Saturation temperature, K.

    WARNING for blends: a zeotropic mixture has no single saturation
    temperature. This returns the BUBBLE point. Use glide() to see the spread.
    """
    return PropsSI("T", "P", p, "Q", 0, fluid)


def glide(fluid: str, p: float) -> float:
    """Dew minus bubble temperature at p, K. Zero for a pure fluid."""
    return (PropsSI("T", "P", p, "Q", 1, fluid)
            - PropsSI("T", "P", p, "Q", 0, fluid))


def h_fg(fluid: str, *, T=None, p=None) -> float:
    """Latent heat, J/kg."""
    return sat_vapour(fluid, T=T, p=p).h - sat_liquid(fluid, T=T, p=p).h


def critical(fluid: str) -> dict:
    """Critical point, for checking you are not extrapolating past it."""
    return {"T": PropsSI("TCRIT", fluid), "p": PropsSI("PCRIT", fluid)}


def fluids() -> list:
    """Every fluid CoolProp knows. There are about 130."""
    import CoolProp
    return sorted(CoolProp.__fluids__)


# ------------------------------------------------------------------ solvers
def solve(f, x0, *, tol=1e-10, max_iter=200, bracket=None):
    """Find x where f(x) = 0.

    Uses Brent's method when you give a bracket (robust, always converges if
    the bracket is valid), otherwise secant from x0. Raises with a readable
    message rather than returning a wrong answer silently, which is the whole
    problem with doing this in a spreadsheet.
    """
    from scipy.optimize import brentq, newton
    if bracket is not None:
        a, b = bracket
        fa, fb = f(a), f(b)
        if fa * fb > 0:
            raise ValueError(
                f"f({a:g})={fa:g} and f({b:g})={fb:g} have the same sign, so no "
                "root is bracketed. Widen the bracket or check the equation."
            )
        return brentq(f, a, b, xtol=tol, maxiter=max_iter)
    return newton(f, x0, tol=tol, maxiter=max_iter)


def sweep(fn, values, *, name="x"):
    """Run fn(v) for each v and collect the results as a list of dicts.

    fn must return a dict. The sweep variable is added under `name`, so the
    result drops straight into to_excel().
    """
    rows = []
    for v in values:
        out = fn(v)
        if not isinstance(out, dict):
            raise TypeError("the swept function must return a dict of results")
        rows.append({name: v, **out})
    return rows


# -------------------------------------------------------------------- excel
def to_excel(path, sheets: dict, *, sources=None, summary=None, title=None):
    """Write a workbook that is genuinely usable.

    Every numeric cell is written as a number (not text), AutoFilter is on,
    the header row is frozen, and columns are sized to content. A Summary and
    a Sources sheet are always present, because a result you cannot trace is
    not an engineering deliverable.

        sheets  = {"Sweep": [ {...}, {...} ], ...}   list of dicts per sheet
        sources = [ ("what", "where it came from"), ... ]
        summary = [ ("quantity", value, "units"), ... ]
    """
    from openpyxl import Workbook
    from openpyxl.styles import Font, PatternFill, Alignment
    from openpyxl.utils import get_column_letter

    wb = Workbook()
    wb.remove(wb.active)
    head_font = Font(bold=True, color="FFFFFF", name="Calibri")
    head_fill = PatternFill("solid", fgColor="1F3864")

    def _write(ws, rows, headers=None):
        headers = headers or (list(rows[0].keys()) if rows else [])
        for j, hname in enumerate(headers, 1):
            c = ws.cell(row=1, column=j, value=hname)
            c.font, c.fill = head_font, head_fill
            c.alignment = Alignment(horizontal="left")
        for i, row in enumerate(rows, 2):
            for j, hname in enumerate(headers, 1):
                v = row.get(hname)
                # Numbers stay numbers. This is the single most common way a
                # delivered workbook turns out not to be filterable.
                if isinstance(v, bool):
                    v = str(v)
                elif isinstance(v, (int, float)) and not isinstance(v, bool):
                    v = float(v) if isinstance(v, float) else v
                ws.cell(row=i, column=j, value=v)
        if rows:
            ws.auto_filter.ref = (f"A1:{get_column_letter(len(headers))}"
                                  f"{len(rows) + 1}")
        ws.freeze_panes = "A2"
        for j, hname in enumerate(headers, 1):
            width = max([len(str(hname))] +
                        [len(f"{r.get(hname)}") for r in rows[:200]]) + 3
            ws.column_dimensions[get_column_letter(j)].width = min(width, 42)

    # Summary first, so it is what opens.
    ws = wb.create_sheet("Summary")
    ws["A1"] = title or "AM5061 results"
    ws["A1"].font = Font(bold=True, size=14, color="1F3864")
    r = 3
    for item in (summary or []):
        for j, v in enumerate(item, 1):
            ws.cell(row=r, column=j, value=v)
        r += 1
    ws.column_dimensions["A"].width = 46
    ws.column_dimensions["B"].width = 18
    ws.column_dimensions["C"].width = 14

    for sname, rows in sheets.items():
        _write(wb.create_sheet(sname[:31]), rows)

    ws = wb.create_sheet("Sources")
    _write(ws, [{"item": a, "source": b} for a, b in (sources or [])])

    wb.save(path)
    return path


# --------------------------------------------------------------- plot style
def style_plots():
    """Match the lecture decks, so figures in a report look like the slides."""
    import matplotlib as mpl
    mpl.rcParams.update({
        "figure.figsize": (7.2, 4.4), "figure.dpi": 110,
        "axes.edgecolor": MUTED, "axes.labelcolor": NAVY,
        "axes.titlecolor": NAVY, "axes.titlesize": 11.5,
        "axes.spines.top": False, "axes.spines.right": False,
        "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
        "xtick.color": MUTED, "ytick.color": MUTED,
        "font.size": 10, "legend.frameon": False,
        "axes.prop_cycle": mpl.cycler(color=[NAVY, ORANGE, BLUE, "#7F9DB9"]),
    })


---
## The case

A 500 TR campus chilled-water plant. The design intent was **69.9 kg/s** through
the DN 200 header. The installed pump is delivering considerably more, and the
energy manager wants to know why, and what it is costing.

Deliverable **D-2**: find the operating point, then compare **throttling**
against **speed control** at the same flow.

### What you are actually doing

The operating point is where the **pump curve** meets the **system curve**.
That is one nonlinear equation in one unknown. In Modelica the solver found it
for you; here you find it yourself, which means you can also draw it.


## 1. The two curves

In [ ]:
import am5061 as am
import numpy as np, matplotlib.pyplot as plt
from scipy.optimize import brentq
am.style_plots()

rho, cp = 995.6, 4181.0        # chilled water, 7 C

dp_0 = 5.05e5    # Pa      pump shut-off head at full speed
m_0  = 140.0     # kg/s    pump runout flow at full speed
k_sys = 59.7     # Pa.s2/kg2   system resistance, as installed

def pump_head(m, N=1.0):
    """Quadratic pump curve, scaled by the affinity laws.
    head ~ N^2 and flow ~ N, so the whole curve slides down-left with speed."""
    return dp_0 * N**2 * (1 - (m / (m_0 * N))**2)

def system_loss(m, k=k_sys):
    """Fully turbulent: loss goes as the square of flow. The sign of m is kept
    so the curve is still correct if you ever reverse the flow."""
    return k * m * abs(m)


## 2. The operating point

Where the two curves cross. `brentq` needs a bracket, and the physics gives you
one: flow is between zero and the pump's runout.


In [ ]:
def operating_point(k=k_sys, N=1.0):
    m = brentq(lambda m: pump_head(m, N) - system_loss(m, k), 1e-6, m_0*N*0.999)
    dp = pump_head(m, N)
    return {"m_dot, kg/s": m, "head, Pa": dp,
            "P_hyd, W": dp*m/rho, "N_rel": N, "k_system": k}

op = operating_point()
for key, val in op.items():
    print(f"  {key:14s} {val:12.4f}")
print(f"\n  design intent was 69.9 kg/s -> running "
      f"{(op['m_dot, kg/s']/69.9 - 1)*100:.1f}% over")


> **Check.** The installed plant settles at **76.8689 kg/s**. That
> overconsumption is what the brief asks you to explain.


## 3. Draw it

If you cannot draw this, you do not understand it.

In [ ]:
m = np.linspace(0, m_0*0.999, 400)
fig, ax = plt.subplots()
ax.plot(m, pump_head(m)/1e5, lw=2.4, label="pump curve, N = 1.0")
ax.plot(m, system_loss(m)/1e5, lw=2.4, label=f"system curve, k = {k_sys}")
ax.plot(m, system_loss(m, k=90)/1e5, lw=1.6, ls="--",
        color=am.MUTED, label="system curve, throttled to k = 90")
ax.plot(m, pump_head(m, N=0.85)/1e5, lw=1.6, ls=":",
        color=am.MUTED, label="pump curve, N = 0.85")
ax.plot(op["m_dot, kg/s"], op["head, Pa"]/1e5, "o", ms=10, color=am.ORANGE, zorder=5)
ax.annotate(f"  {op['m_dot, kg/s']:.2f} kg/s", (op["m_dot, kg/s"], op["head, Pa"]/1e5),
            color=am.ORANGE, fontsize=11, va="center")
ax.axvline(69.9, color=am.MUTED, lw=1, ls="-.")
ax.text(69.9, 5.3, " design intent", color=am.MUTED, fontsize=9)
ax.set_xlabel("mass flow  (kg/s)"); ax.set_ylabel("head  (bar)")
ax.set_title("The operating point is an intersection, not a specification")
ax.set_ylim(0, 5.6); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()


## 4. The design question

Get back to the design flow of **69.9 kg/s** two ways, and compare the power.

- **Throttle**: close a valve, which raises `k_system`.
- **Slow down**: turn the VFD down, which lowers `N_rel`.

Both reach the same flow. They do not cost the same.


In [ ]:
TARGET = 69.9

k_throttled = brentq(lambda k: operating_point(k=k)["m_dot, kg/s"] - TARGET, 30., 400.)
N_slowed    = brentq(lambda N: operating_point(N=N)["m_dot, kg/s"] - TARGET, 0.3, 1.2)

cases = {"as installed": operating_point(),
         f"throttled (k={k_throttled:.1f})": operating_point(k=k_throttled),
         f"VFD (N={N_slowed:.4f})":          operating_point(N=N_slowed)}

print(f"{'':28s}{'flow kg/s':>12s}{'head bar':>11s}{'P_hyd kW':>11s}")
for nme, c in cases.items():
    print(f"{nme:28s}{c['m_dot, kg/s']:12.3f}{c['head, Pa']/1e5:11.3f}{c['P_hyd, W']/1e3:11.3f}")

p_t = cases[f"throttled (k={k_throttled:.1f})"]["P_hyd, W"]
p_v = cases[f"VFD (N={N_slowed:.4f})"]["P_hyd, W"]
print(f"\n  speed control saves {(p_t-p_v)/p_t*100:.1f}% of hydraulic power "
      f"at the SAME flow ({(p_t-p_v)/1e3:.2f} kW)")
print("  Throttling does not remove the energy. It moves it into the valve.")


## 5. The deliverable

In [ ]:
ks = np.arange(40., 161., 5.)
rows_k = am.sweep(lambda k: operating_point(k=k), ks, name="k_system")
Ns = np.arange(0.60, 1.05, 0.025)
rows_N = am.sweep(lambda N: operating_point(N=N), Ns, name="N_rel_swept")

path = am.to_excel("AM5061_D2_Hydraulics.xlsx",
    {"Throttling": rows_k, "Speed control": rows_N},
    title="AM5061 D-2 . 500 TR campus chilled-water distribution",
    summary=[("Design intent flow", 69.9, "kg/s"),
             ("As-installed flow", op["m_dot, kg/s"], "kg/s"),
             ("Pump shut-off head", dp_0, "Pa"),
             ("Pump runout flow", m_0, "kg/s"),
             ("System resistance as installed", k_sys, "Pa.s2/kg2"),
             ("Hydraulic power saved by VFD at design flow", (p_t-p_v), "W")],
    sources=[("Water properties", "rho 995.6 kg/m3, cp 4181 J/kgK at 7 C"),
             ("Pump curve", "Quadratic fit to the installed pump, AM5061 brief D-2"),
             ("System curve", "Fully turbulent, dp = k m|m|")])
print("written:", path)


## What to hand in

1. The operating point, with the intersection plot.
2. Why the plant runs over its design flow.
3. Throttling versus speed control at 69.9 kg/s: flow, head and hydraulic power
   for each, and the annual cost difference at your own tariff assumption.
4. The workbook.

**One paragraph:** where does the energy go when you throttle?
